# Notebook 03 — Train / Validation / Test Split

This notebook reads the labeled order table created by Notebook 02 and splits it into training, validation, and test sets.

Goals:
- Read and validate the labeled artifact
- Inspect the time range and target distribution
- Compare random and time-based splitting conceptually
- Use a chronological 70/15/15 split
- Validate that the splits do not overlap
- Validate chronological ordering
- Inspect label balance across splits
- Save train, validation, and test artifacts

Artifacts:
- `artifacts/03_splits/train.parquet`
- `artifacts/03_splits/validation.parquet`
- `artifacts/03_splits/test.parquet`

The test set must remain untouched after this notebook until final model evaluation.

In [1]:
from pathlib import Path

import pandas as pd


pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.width",
    220
)


def find_project_root():

    current_path = Path.cwd().resolve()

    candidate_roots = [
        current_path,
        *current_path.parents,
    ]

    for candidate in candidate_roots:

        if (
            (candidate / "compose.yaml").exists()
            and
            (candidate / "requirements.txt").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the Olist-MLOps project root. "
        "Run Jupyter from inside the project directory."
    )


PROJECT_ROOT = find_project_root()


INPUT_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "02_labeled"
    / "labeled_table.parquet"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "03_splits"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


assert INPUT_PATH.exists(), (
    f"Notebook 02 artifact not found: {INPUT_PATH}"
)


print(
    "Project root :",
    PROJECT_ROOT
)

print(
    "Input        :",
    INPUT_PATH
)

print(
    "Input exists :",
    INPUT_PATH.exists()
)

print(
    "Output       :",
    OUTPUT_DIR
)

Project root : G:\(01)04\Qafza_MLOps\Olist-MLOps
Input        : G:\(01)04\Qafza_MLOps\Olist-MLOps\artifacts\02_labeled\labeled_table.parquet
Input exists : True
Output       : G:\(01)04\Qafza_MLOps\Olist-MLOps\artifacts\03_splits


In [2]:
df = pd.read_parquet(
    INPUT_PATH
)

print(
    "Labeled table loaded successfully."
)

print(
    "Shape:",
    df.shape
)

Labeled table loaded successfully.
Shape: (96470, 46)


In [3]:
print(
    "Rows:",
    len(df)
)

print(
    "Columns:",
    df.shape[1]
)

print(
    "Unique order_id:",
    df["order_id"].nunique()
)

print(
    "Duplicate order_id:",
    df["order_id"]
    .duplicated()
    .sum()
)

print(
    "Missing labels:",
    df["is_late"]
    .isna()
    .sum()
)

assert (
    df["order_id"].nunique()
    == len(df)
)

assert (
    df["order_id"]
    .duplicated()
    .sum()
    == 0
)

assert (
    df["is_late"]
    .isna()
    .sum()
    == 0
)

assert set(
    df["is_late"].unique()
) == {0, 1}

print(
    "\nInput artifact validation passed."
)

Rows: 96470
Columns: 46
Unique order_id: 96470
Duplicate order_id: 0
Missing labels: 0

Input artifact validation passed.


In [4]:
SPLIT_DATE_COLUMN = (
    "order_purchase_timestamp"
)

assert (
    SPLIT_DATE_COLUMN
    in df.columns
)

df[
    SPLIT_DATE_COLUMN
] = pd.to_datetime(
    df[
        SPLIT_DATE_COLUMN
    ],
    errors="coerce"
)

missing_split_dates = (
    df[
        SPLIT_DATE_COLUMN
    ]
    .isna()
    .sum()
)

print(
    "Split date column:",
    SPLIT_DATE_COLUMN
)

print(
    "Missing split dates:",
    missing_split_dates
)

assert (
    missing_split_dates == 0
)

Split date column: order_purchase_timestamp
Missing split dates: 0


In [5]:
earliest_order = (
    df[
        SPLIT_DATE_COLUMN
    ].min()
)

latest_order = (
    df[
        SPLIT_DATE_COLUMN
    ].max()
)

print(
    "Earliest order:",
    earliest_order
)

print(
    "Latest order  :",
    latest_order
)

Earliest order: 2016-09-15 12:16:38
Latest order  : 2018-08-29 15:00:37


In [6]:
df[
    "purchase_month"
] = (
    df[
        SPLIT_DATE_COLUMN
    ]
    .dt.to_period("M")
)

monthly_counts = (
    df
    .groupby(
        "purchase_month"
    )
    .size()
    .rename("orders")
)

display(
    monthly_counts
)

purchase_month
2016-09       1
2016-10     265
2016-12       1
2017-01     750
2017-02    1653
2017-03    2546
2017-04    2303
2017-05    3545
2017-06    3135
2017-07    3872
2017-08    4193
2017-09    4150
2017-10    4478
2017-11    7288
2017-12    5513
2018-01    7069
2018-02    6555
2018-03    7003
2018-04    6798
2018-05    6749
2018-06    6096
2018-07    6156
2018-08    6351
Freq: M, Name: orders, dtype: int64

In [7]:
monthly_label = (
    df
    .groupby(
        "purchase_month"
    )
    .agg(
        orders=(
            "order_id",
            "count"
        ),
        late_orders=(
            "is_late",
            "sum"
        ),
        late_rate=(
            "is_late",
            "mean"
        ),
    )
)

monthly_label[
    "late_rate_pct"
] = (
    monthly_label[
        "late_rate"
    ]
    * 100
)

display(
    monthly_label[
        [
            "orders",
            "late_orders",
            "late_rate_pct",
        ]
    ]
)

,orders,late_orders,late_rate_pct
purchase_month,,,
2016-09,1,1,100.000000
2016-10,265,2,0.754717
2016-12,1,0,0.000000
2017-01,750,22,2.933333
2017-02,1653,49,2.964307
2017-03,2546,116,4.556167
2017-04,2303,151,6.556665
2017-05,3545,106,2.990127
2017-06,3135,95,3.030303


## Random Split vs Time-Based Split

Two common options are considered:

### Random Split

A random split mixes orders from different time periods across training, validation, and test sets.

Advantages:
- Usually maintains similar target proportions across splits
- Simple to implement

Disadvantage for this project:
- Future-period orders could influence training while earlier orders appear in validation or test
- This does not closely represent how the model would be used in production

### Time-Based Split

A time-based split trains the model on older orders and evaluates it on newer orders.

Advantages:
- Better represents the real prediction scenario
- Prevents future orders from appearing in the training set
- Tests whether the model generalizes to later periods

The monthly late-delivery rate changes over time in this dataset, which makes chronological evaluation particularly useful.

In [8]:
sorted_df = (
    df
    .sort_values(
        SPLIT_DATE_COLUMN
    )
    .reset_index(
        drop=True
    )
)

n = len(sorted_df)

train_end_index = int(
    n * 0.70
)

val_end_index = int(
    n * 0.85
)

candidate_train = (
    sorted_df.iloc[
        :train_end_index
    ]
)

candidate_val = (
    sorted_df.iloc[
        train_end_index:
        val_end_index
    ]
)

candidate_test = (
    sorted_df.iloc[
        val_end_index:
    ]
)

print(
    "Total rows :",
    n
)

print(
    "Train      :",
    len(candidate_train)
)

print(
    "Validation :",
    len(candidate_val)
)

print(
    "Test       :",
    len(candidate_test)
)

Total rows : 96470
Train      : 67529
Validation : 14470
Test       : 14471


In [9]:
def split_summary(
    name,
    data
):

    late_count = int(
        data[
            "is_late"
        ].sum()
    )

    return {
        "split":
            name,

        "rows":
            len(data),

        "percentage":
            len(data)
            / len(df)
            * 100,

        "start_date":
            data[
                SPLIT_DATE_COLUMN
            ].min(),

        "end_date":
            data[
                SPLIT_DATE_COLUMN
            ].max(),

        "on_time_orders":
            len(data)
            - late_count,

        "late_orders":
            late_count,

        "late_rate_pct":
            data[
                "is_late"
            ].mean()
            * 100,
    }


candidate_summary = pd.DataFrame([
    split_summary(
        "Train",
        candidate_train
    ),

    split_summary(
        "Validation",
        candidate_val
    ),

    split_summary(
        "Test",
        candidate_test
    ),
])

display(
    candidate_summary
)

,split,rows,percentage,start_date,end_date,on_time_orders,late_orders,late_rate_pct
0,Train,67529,70.000000,2016-09-15 12:16:38,2018-04-15 20:12:35,62239,5290,7.833671
1,Validation,14470,14.999482,2018-04-15 20:17:11,2018-06-21 08:29:29,13846,624,4.312370
2,Test,14471,15.000518,2018-06-21 08:41:07,2018-08-29 15:00:37,13851,620,4.284431


## Final Split Strategy

A time-based 70/15/15 split is selected.

The model is intended to learn from historical orders and make predictions for future orders. Therefore, chronological splitting better represents the real deployment scenario than a random split.

Final strategy:

- **Train:** earliest 70% of labeled orders
- **Validation:** following 15%
- **Test:** latest 15%

The late-delivery rate differs between historical periods, so the validation and test sets also measure how well the model handles temporal distribution shift.

After this notebook:

- Training data may be used for EDA and feature engineering
- Validation data may be used during model development
- Test data must remain untouched until final model evaluation

In [10]:
final_df = (
    sorted_df
    .drop(
        columns=[
            "purchase_month"
        ],
        errors="ignore"
    )
)

n = len(final_df)

train_end_index = int(
    n * 0.70
)

val_end_index = int(
    n * 0.85
)

train_df = (
    final_df.iloc[
        :train_end_index
    ]
    .copy()
)

val_df = (
    final_df.iloc[
        train_end_index:
        val_end_index
    ]
    .copy()
)

test_df = (
    final_df.iloc[
        val_end_index:
    ]
    .copy()
)

print(
    "Train      :",
    train_df.shape
)

print(
    "Validation :",
    val_df.shape
)

print(
    "Test       :",
    test_df.shape
)

Train      : (67529, 46)
Validation : (14470, 46)
Test       : (14471, 46)


In [11]:
total_split_rows = (
    len(train_df)
    + len(val_df)
    + len(test_df)
)

print(
    "Original rows:",
    len(df)
)

print(
    "Split rows   :",
    total_split_rows
)

assert (
    total_split_rows
    == len(df)
)

assert len(train_df) > 0
assert len(val_df) > 0
assert len(test_df) > 0

print(
    "\nRow-count validation passed."
)

Original rows: 96470
Split rows   : 96470

Row-count validation passed.


In [12]:
train_ids = set(
    train_df[
        "order_id"
    ]
)

val_ids = set(
    val_df[
        "order_id"
    ]
)

test_ids = set(
    test_df[
        "order_id"
    ]
)

train_val_overlap = (
    train_ids
    & val_ids
)

train_test_overlap = (
    train_ids
    & test_ids
)

val_test_overlap = (
    val_ids
    & test_ids
)

print(
    "Train ↔ Validation overlap:",
    len(train_val_overlap)
)

print(
    "Train ↔ Test overlap:",
    len(train_test_overlap)
)

print(
    "Validation ↔ Test overlap:",
    len(val_test_overlap)
)

assert (
    len(train_val_overlap)
    == 0
)

assert (
    len(train_test_overlap)
    == 0
)

assert (
    len(val_test_overlap)
    == 0
)

print(
    "\nNo order overlap between splits."
)

Train ↔ Validation overlap: 0
Train ↔ Test overlap: 0
Validation ↔ Test overlap: 0

No order overlap between splits.


In [13]:
train_max_date = (
    train_df[
        SPLIT_DATE_COLUMN
    ].max()
)

val_min_date = (
    val_df[
        SPLIT_DATE_COLUMN
    ].min()
)

val_max_date = (
    val_df[
        SPLIT_DATE_COLUMN
    ].max()
)

test_min_date = (
    test_df[
        SPLIT_DATE_COLUMN
    ].min()
)

print(
    "Train max date      :",
    train_max_date
)

print(
    "Validation min date :",
    val_min_date
)

print()

print(
    "Validation max date :",
    val_max_date
)

print(
    "Test min date       :",
    test_min_date
)

assert (
    train_max_date
    <= val_min_date
)

assert (
    val_max_date
    <= test_min_date
)

print(
    "\nChronological validation passed."
)

Train max date      : 2018-04-15 20:12:35
Validation min date : 2018-04-15 20:17:11

Validation max date : 2018-06-21 08:29:29
Test min date       : 2018-06-21 08:41:07

Chronological validation passed.


In [14]:
final_summary = pd.DataFrame([
    split_summary(
        "Train",
        train_df
    ),

    split_summary(
        "Validation",
        val_df
    ),

    split_summary(
        "Test",
        test_df
    ),
])

display(
    final_summary
)

,split,rows,percentage,start_date,end_date,on_time_orders,late_orders,late_rate_pct
0,Train,67529,70.000000,2016-09-15 12:16:38,2018-04-15 20:12:35,62239,5290,7.833671
1,Validation,14470,14.999482,2018-04-15 20:17:11,2018-06-21 08:29:29,13846,624,4.312370
2,Test,14471,15.000518,2018-06-21 08:41:07,2018-08-29 15:00:37,13851,620,4.284431


In [15]:
for name, data in {
    "Train":
        train_df,

    "Validation":
        val_df,

    "Test":
        test_df,
}.items():

    rows = len(data)

    unique_orders = (
        data[
            "order_id"
        ].nunique()
    )

    duplicate_orders = (
        data[
            "order_id"
        ]
        .duplicated()
        .sum()
    )

    missing_labels = (
        data[
            "is_late"
        ]
        .isna()
        .sum()
    )

    missing_dates = (
        data[
            SPLIT_DATE_COLUMN
        ]
        .isna()
        .sum()
    )

    print(
        f"{name:<12} "
        f"rows={rows:,} | "
        f"unique_orders={unique_orders:,} | "
        f"duplicates={duplicate_orders} | "
        f"missing_labels={missing_labels} | "
        f"missing_dates={missing_dates}"
    )

    assert (
        rows
        == unique_orders
    )

    assert (
        duplicate_orders
        == 0
    )

    assert (
        missing_labels
        == 0
    )

    assert (
        missing_dates
        == 0
    )

print(
    "\nAll split-level validations passed."
)

Train        rows=67,529 | unique_orders=67,529 | duplicates=0 | missing_labels=0 | missing_dates=0
Validation   rows=14,470 | unique_orders=14,470 | duplicates=0 | missing_labels=0 | missing_dates=0
Test         rows=14,471 | unique_orders=14,471 | duplicates=0 | missing_labels=0 | missing_dates=0

All split-level validations passed.


In [16]:
TRAIN_PATH = (
    OUTPUT_DIR
    / "train.parquet"
)

VAL_PATH = (
    OUTPUT_DIR
    / "validation.parquet"
)

TEST_PATH = (
    OUTPUT_DIR
    / "test.parquet"
)

train_df.to_parquet(
    TRAIN_PATH,
    index=False
)

val_df.to_parquet(
    VAL_PATH,
    index=False
)

test_df.to_parquet(
    TEST_PATH,
    index=False
)

print(
    "Saved:"
)

print(
    "Train      :",
    TRAIN_PATH
)

print(
    "Validation :",
    VAL_PATH
)

print(
    "Test       :",
    TEST_PATH
)

Saved:
Train      : G:\(01)04\Qafza_MLOps\Olist-MLOps\artifacts\03_splits\train.parquet
Validation : G:\(01)04\Qafza_MLOps\Olist-MLOps\artifacts\03_splits\validation.parquet
Test       : G:\(01)04\Qafza_MLOps\Olist-MLOps\artifacts\03_splits\test.parquet


In [17]:
saved_train = pd.read_parquet(
    TRAIN_PATH
)

saved_val = pd.read_parquet(
    VAL_PATH
)

saved_test = pd.read_parquet(
    TEST_PATH
)

print(
    "Saved Train      :",
    saved_train.shape
)

print(
    "Saved Validation :",
    saved_val.shape
)

print(
    "Saved Test       :",
    saved_test.shape
)

assert (
    saved_train.shape
    == train_df.shape
)

assert (
    saved_val.shape
    == val_df.shape
)

assert (
    saved_test.shape
    == test_df.shape
)

assert (
    saved_train[
        "order_id"
    ]
    .duplicated()
    .sum()
    == 0
)

assert (
    saved_val[
        "order_id"
    ]
    .duplicated()
    .sum()
    == 0
)

assert (
    saved_test[
        "order_id"
    ]
    .duplicated()
    .sum()
    == 0
)

assert (
    saved_train[
        "is_late"
    ]
    .isna()
    .sum()
    == 0
)

assert (
    saved_val[
        "is_late"
    ]
    .isna()
    .sum()
    == 0
)

assert (
    saved_test[
        "is_late"
    ]
    .isna()
    .sum()
    == 0
)

print(
    "\nAll saved split artifacts validated successfully."
)

Saved Train      : (67529, 46)
Saved Validation : (14470, 46)
Saved Test       : (14471, 46)

All saved split artifacts validated successfully.


In [18]:
print(
    "=" * 65
)

print(
    "NOTEBOOK 03 COMPLETE"
)

print(
    "=" * 65
)

print(
    f"Input labeled orders : "
    f"{len(df):,}"
)

print()

print(
    f"Train                : "
    f"{len(saved_train):,} "
    f"({len(saved_train) / len(df) * 100:.2f}%)"
)

print(
    f"Validation           : "
    f"{len(saved_val):,} "
    f"({len(saved_val) / len(df) * 100:.2f}%)"
)

print(
    f"Test                 : "
    f"{len(saved_test):,} "
    f"({len(saved_test) / len(df) * 100:.2f}%)"
)

print()

print(
    f"Train late rate      : "
    f"{saved_train['is_late'].mean() * 100:.2f}%"
)

print(
    f"Validation late rate : "
    f"{saved_val['is_late'].mean() * 100:.2f}%"
)

print(
    f"Test late rate       : "
    f"{saved_test['is_late'].mean() * 100:.2f}%"
)

print()

print(
    "Split strategy       : "
    "Time-based 70/15/15"
)

print(
    "Order overlap        : 0"
)

print(
    "Chronological order  : Valid"
)

print()

print(
    f"Train artifact       : {TRAIN_PATH}"
)

print(
    f"Validation artifact  : {VAL_PATH}"
)

print(
    f"Test artifact        : {TEST_PATH}"
)

NOTEBOOK 03 COMPLETE
Input labeled orders : 96,470

Train                : 67,529 (70.00%)
Validation           : 14,470 (15.00%)
Test                 : 14,471 (15.00%)

Train late rate      : 7.83%
Validation late rate : 4.31%
Test late rate       : 4.28%

Split strategy       : Time-based 70/15/15
Order overlap        : 0
Chronological order  : Valid

Train artifact       : G:\(01)04\Qafza_MLOps\Olist-MLOps\artifacts\03_splits\train.parquet
Validation artifact  : G:\(01)04\Qafza_MLOps\Olist-MLOps\artifacts\03_splits\validation.parquet
Test artifact        : G:\(01)04\Qafza_MLOps\Olist-MLOps\artifacts\03_splits\test.parquet
